In [20]:
import geopandas as gpd
import pandas as pd

In [10]:
# cleaning the dataset
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/Nuclear-Map/data/raw/nuclear_power_plants.csv")
print(df.columns)

Index(['Id', 'Name', 'Latitude', 'Longitude', 'Country', 'CountryCode',
       'Status', 'ReactorType', 'ReactorModel', 'ConstructionStartAt',
       'OperationalFrom', 'OperationalTo', 'Capacity', 'LastUpdatedAt',
       'Source', 'IAEAId'],
      dtype='object')


In [28]:
df["Latitude"]  = pd.to_numeric(df["Latitude"],  errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# Step 2 — drop missing coordinates
before = len(df)
df = df.dropna(subset=["Latitude", "Longitude"])
after = len(df)

In [ ]:
# Filling missing values with "NA"
df = df.fillna("NA")
# Dropping unnecessary columns
df = df.drop(columns=["data_from_source", "last_updated_at", "iaea_id"], errors="ignore")

In [22]:
# Only keep operational and under construction plants  
print(df['Status'].unique())
operating_plants = df[df["Status"] == "Operational"]
underconstruction_plants = df[df["Status"] == "Under Construction"]
print(len(operating_plants))
print(len(underconstruction_plants))

positive_sites = pd.concat([operating_plants, underconstruction_plants])

positive_sites.to_csv("/Users/mohammadbilal/Documents/Projects/Nuclear-Map/data/processed/positive_sites.csv", index=False)
operating_plants.to_csv("/Users/mohammadbilal/Documents/Projects/Nuclear-Map/data/processed/operating_plants.csv", index=False)

['Shutdown' 'Operational' 'Planned' 'Under Construction'
 'Cancelled Construction' 'Suspended Construction' 'Unknown'
 'Decommissioning Completed' 'Suspended Operation' 'Never Commissioned']
411
60


In [30]:
# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.Longitude, df.Latitude),
    crs="EPSG:4326"
)

In [33]:
# Save Datasets
gdf.to_file("../data/processed/nuclear_plants.geojson", driver="GeoJSON")
df.to_csv("../data/processed/nuclear_plants.csv", index=False)